# Shape: generate a domain into lakehouse Delta tables (Python notebook)

Kernel: **Python 3.11 or 3.12** (not PySpark). Generates every table of an installed domain at a
scale preset with a seed, and writes each as a Delta table `Tables/<tablePrefix><table>` in the
default lakehouse. The same seed always generates the same rows.

Next to the tables it writes `Files/<outputDir>/<domain>/contract.json`, the contract that the
domain's own schema implies for these tables (row counts, columns, types, nullability, primary
keys, enumerated values), and `generation.json`. `shape_profile_domain.ipynb` profiles the
tables and checks them against that contract; `shape_generate_gate` runs both from a pipeline.

The exit value is `{domain, scale, seed, mode, tablePrefix, tables, totalRows, contractPath,
kernel}`. `notebookutils.notebook.exit` is the last statement, outside any `try`/`except`.


In [ ]:
%%configure
{"vCores": 8}


In [ ]:
# Upload the wheel(s) to this notebook's built-in resources folder (Resources > builtin):
# the platform wheel (Rust kernel), sqllocks_shape-0.9.0-py3-none-any.whl (pure Python) and, for a domain,
# sqllocks_shape_domains-0.9.0-py3-none-any.whl. pip takes the platform wheel when one
# fits and falls back to the pure wheel; the exit value reports the kernel.
# Once the packages are on PyPI the same line works without the upload.
%pip install --find-links builtin "sqllocks-shape==0.9.0" "sqllocks-shape-domains==0.9.0"

In [ ]:
# Parameters cell (toggle "parameter cell" in Fabric). A pipeline overrides these.
domain = "retail"  # an installed domain (`shape list`)
scale = "small"  # a scale preset of the domain (`shape presets`)
seed = 42  # the same seed always generates the same rows
mode = ""  # "3nf" or "star"; empty keeps the domain's default schema
tablePrefix = ""  # Delta tables are named <tablePrefix><table>, e.g. "retail_"
writeMode = "overwrite"  # "overwrite" or "append" for the Delta tables
outputDir = "shape"  # the contract and manifest go to Files/<outputDir>/<domain>/

In [ ]:
import json
from pathlib import Path

import shape
from shape.integrations.fabric import generation
from shape.kernel.dispatch import get_kernel

KERNEL = get_kernel().NAME  # "rust" with a platform wheel, "python" with the pure-Python wheel
print(f"Shape {shape.__version__}, kernel: {KERNEL}")

LAKEHOUSE = "/lakehouse/default"
FILES = f"{LAKEHOUSE}/Files"
TABLES = f"{LAKEHOUSE}/Tables"
MAX_LISTED = 100

# Names reach paths and table names: only identifiers are accepted.
for _label, _value in (("domain", domain), ("scale", scale), ("outputDir", outputDir)):
    generation.check_name(str(_value).strip("/"), _label)
if tablePrefix:
    generation.check_name(str(tablePrefix), "tablePrefix")

In [ ]:
result = generation.generate_domain(
    str(domain),
    scale=str(scale),
    seed=int(seed),
    mode=str(mode) or None,
)
row_counts = {name: table.num_rows for name, table in result.tables.items()}
print(f"Generated {sum(row_counts.values()):,} rows in {len(row_counts)} tables")
for name in result.generation_order:
    print(f"  {name:<24} {row_counts[name]:>10,} rows")

In [ ]:
tables = generation.write_delta_tables(result, TABLES, prefix=str(tablePrefix), mode=str(writeMode))

# The contract the domain's own schema implies for these tables: the pipeline checks the
# profile of the Delta tables against it (shape_profile_domain.ipynb).
# It expects the rows the schema and scale plan, so a short table fails it.
planned = generation.plan_row_counts(str(domain), str(scale), str(mode) or None)
contract = generation.domain_contract(result.schema, planned)
contract_rel = f"{str(outputDir).strip('/')}/{domain}/contract.json"
generation.write_contract(contract, Path(FILES) / contract_rel)
manifest = {
    "domain": domain,
    "scale": scale,
    "seed": int(seed),
    "mode": str(mode) or "default",
    "tablePrefix": str(tablePrefix),
    "tables": tables,
}
(Path(FILES) / contract_rel).with_name("generation.json").write_text(
    json.dumps(manifest, indent=1), encoding="utf-8"
)
print("Delta tables written under", TABLES)
print("Contract written to", Path(FILES) / contract_rel)

In [ ]:
result_value = {
    "domain": domain,
    "scale": scale,
    "seed": int(seed),
    "mode": str(mode) or "default",
    "tablePrefix": str(tablePrefix),
    "tables": tables[:MAX_LISTED],
    "totalRows": sum(t["rows"] for t in tables),
    "contractPath": contract_rel,
    "kernel": KERNEL,
}
print(json.dumps(result_value, indent=2)[:4000])

In [ ]:
import notebookutils

notebookutils.notebook.exit(json.dumps(result_value))